# TP1 — Comprendre à qui on parle et de quoi on dispose

**Cas d'usage 01 — Certification IA : Résiliation client SaaS (churn)**

| | |
|---|---|
| **Objectif** | Cerner le besoin métier (churn, CLV, rétention) et faire l'inventaire des données disponibles — avant d'écrire la moindre ligne de nettoyage ou de modélisation. |
| **Livrable** | Ce notebook de cadrage : contexte métier, dictionnaire de données, gouvernance, questions de confidentialité. |
| **Enjeu** | Peut-on utiliser toutes ces données sans problème ? Y a-t-il des informations sensibles à traiter avec précaution ? |
| **Compétences** | C1 (identifier un jeu de données pertinent), amorce C2 (risques éthiques) |

> Ce TP correspond aux sections 2-4 du plan imposé par le règlement de la certification
> (`Enonce cas usage_churn_saas.pdf` §3.1) : cadrage métier, données/gouvernance, enjeux
> éthiques. Il ne nettoie ni ne modélise rien — c'est l'objet de TP2 et suivants.

## §1 — Cadrage métier et cas d'usage [C1]

### Le contexte

Un éditeur de logiciel SaaS B2B vend plusieurs formules d'abonnement à des entreprises
clientes. Modèle économique : **revenus récurrents** — chaque mois, le client paie selon
le plan souscrit, le nombre de licences et son niveau d'usage.

### Le churn

Le **churn** est la résiliation d'un client à l'échéance de son contrat — il cesse de
payer et d'utiliser le produit. C'est un indicateur stratégique : il impacte le MRR
(revenu mensuel récurrent), la rentabilité, la croissance commerciale et la valeur vie
client.

- `churn = 0` → le client renouvelle
- `churn = 1` → le client résilie

### La cible secondaire : la CLV

La **Customer Lifetime Value (CLV)** estime le revenu total qu'un client génèrera sur
toute la durée de la relation commerciale. Elle répond à une question différente et
complémentaire de celle du churn :

| Question | Réponse apportée par |
|---|---|
| *Ce client risque-t-il de partir ?* | Le modèle de **classification** (churn) |
| *Que perdrait-on financièrement s'il partait ?* | Le modèle de **régression** (CLV) |

**Règle méthodologique non négociable** : la CLV est une cible secondaire qui enrichit
la priorisation métier (risque élevé **et** forte CLV = urgence maximale) mais ne doit
**jamais** servir de variable explicative au modèle de churn — ce serait injecter dans
le modèle une information dérivée du même phénomène qu'on cherche à prédire, un biais
méthodologique proche d'une fuite de données. Point de vigilance à vérifier explicitement
en TP3 et TP8.

### Ce que le modèle doit produire (au-delà d'une simple probabilité)

1. Un **score de risque** de churn par client.
2. Les **facteurs explicatifs** de ce risque (faible usage, insatisfaction, retards de
   paiement...).
3. Une **priorisation** combinant risque de churn et CLV.
4. Une **recommandation d'action** de rétention adaptée à chaque situation.

### Exemples d'actions de rétention reliées aux signaux disponibles

| Signal détecté | Action de rétention envisageable |
|---|---|
| Faible taux d'adoption | Formation, démonstration personnalisée |
| Peu d'utilisateurs actifs | Accompagnement au déploiement |
| Dernière connexion ancienne | Contact rapide pour comprendre les difficultés |
| Nombreux tickets support | Priorisation des incidents, suivi renforcé |
| CSAT faible | Entretien personnalisé, plan d'action |
| Retards de paiement | Nouvel échéancier, offre adaptée |
| Fonctionnalités peu exploitées | Présentation des usages avancés |

**Journal de bord [C1]** — Le modèle ne remplace pas la décision humaine : il fournit un
outil d'aide à la décision pour que les équipes Customer Success priorisent leurs
efforts. Cette distinction encadrera tous les choix de conception ultérieurs (pas de
rejet/downgrade automatique, seuil de décision pensé comme un déclencheur d'alerte, pas
comme une sentence).

## §2 — Données : disponibilité, gouvernance et alternatives [C1]

### Inventaire des fichiers fournis

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")

FILES = {
    "complet":     DATA_DIR / "churn_saas_complet.csv",
    "echantillon": DATA_DIR / "churn_saas_echantillon.csv",
    "catalogue":   DATA_DIR / "catalogue_plans.csv",
}

for name, path in FILES.items():
    exists = path.exists()
    size_ko = path.stat().st_size / 1024 if exists else 0
    print(f"{name:12s} {path.name:32s} existe={exists}  taille={size_ko:.1f} Ko")


complet      churn_saas_complet.csv           existe=True  taille=710.6 Ko
echantillon  churn_saas_echantillon.csv       existe=True  taille=7.6 Ko
catalogue    catalogue_plans.csv              existe=True  taille=0.2 Ko


In [2]:
df_complet = pd.read_csv(FILES["complet"])
df_echantillon = pd.read_csv(FILES["echantillon"])
df_catalogue = pd.read_csv(FILES["catalogue"])

print(f"complet     : {df_complet.shape[0]:,} lignes x {df_complet.shape[1]} colonnes")
print(f"echantillon : {df_echantillon.shape[0]:,} lignes x {df_echantillon.shape[1]} colonnes")
print(f"catalogue   : {df_catalogue.shape[0]:,} lignes x {df_catalogue.shape[1]} colonnes")
print()
print("Colonnes (complet) :")
print(list(df_complet.columns))


complet     : 5,035 lignes x 29 colonnes
echantillon : 50 lignes x 29 colonnes
catalogue   : 4 lignes x 6 colonnes

Colonnes (complet) :
['client_id', 'date_souscription', 'jour_souscription', 'secteur', 'pays', 'taille_entreprise', 'plan', 'anciennete_mois', 'sieges_souscrits', 'utilisateurs_actifs', 'taux_adoption_pct', 'connexions_30j', 'heures_usage_30j', 'fonctionnalites_total', 'fonctionnalites_utilisees', 'nb_integrations', 'derniere_connexion_jours', 'tickets_support_90j', 'delai_reponse_support_h', 'csat', 'retards_paiement_12m', 'revenu_mensuel_recurrent_eur', 'couleur_theme_interface', 'code_datacenter', 'groupe_experimentation', 'commentaire_csm', 'sante_compte_fin_periode', 'valeur_vie_client_eur', 'churn']


`churn_saas_complet.csv` correspond exactement au dictionnaire de données de l'énoncé
(5035 lignes, 29 colonnes). `churn_saas_echantillon.csv` est un extrait de 50 lignes —
utile pour des vérifications rapides sans recharger le fichier complet.
`catalogue_plans.csv` est un référentiel externe (prix, fonctionnalités, SLA par plan) à
joindre sur la colonne `plan`.

In [3]:
df_catalogue


,plan,prix_mensuel_par_siege_eur,fonctionnalites_incluses,sla_reponse_h,quota_stockage_go,support_dedie
0,Starter,12,8,24,10,Non
1,Pro,25,16,12,100,Non
2,Business,45,26,6,500,Oui
3,Enterprise,80,40,3,2000,Oui


### Dictionnaire de données (résumé)

| Variable | Type | Rôle |
|---|---|---|
| `client_id` | texte | Identifiant — à exclure de toute feature (pas de pouvoir prédictif, juste une clé) |
| `date_souscription`, `jour_souscription` | date / catégoriel | Ancienneté, saisonnalité |
| `secteur`, `pays`, `taille_entreprise`, `plan` | catégoriel | Segmentation contractuelle |
| `anciennete_mois`, `sieges_souscrits`, `utilisateurs_actifs`, `taux_adoption_pct` | numérique | Usage et adoption |
| `connexions_30j`, `heures_usage_30j`, `fonctionnalites_utilisees`, `nb_integrations` | numérique | Engagement produit |
| `derniere_connexion_jours` | numérique | Signal de désengagement |
| `tickets_support_90j`, `delai_reponse_support_h`, `csat` | numérique | Expérience support |
| `retards_paiement_12m`, `revenu_mensuel_recurrent_eur` | numérique | Santé financière du compte |
| `couleur_theme_interface`, `code_datacenter`, `groupe_experimentation` | catégoriel | **Suspects — leurres probables** (à vérifier en TP3) |
| `commentaire_csm` | texte libre | Note qualitative du Customer Success Manager |
| `sante_compte_fin_periode` | numérique | **Suspect — piège de fuite** (calculé en fin de période, à vérifier en TP3) |
| `valeur_vie_client_eur` | numérique | **Cible secondaire (régression)** — jamais en feature du modèle churn |
| `churn` | binaire | **Cible principale (classification)** |

### Gouvernance des données — d'où viennent-elles en pratique ?

Dans une entreprise SaaS réelle, ces informations proviendraient de systèmes distincts,
qu'il faudrait faire dialoguer :

| Donnée | Système source typique |
|---|---|
| Usage, connexions, adoption | Outil d'analytics produit (ex. Mixpanel, Amplitude) |
| Tickets, délai de réponse, CSAT | Outil de support (ex. Zendesk, Intercom) |
| Facturation, retards de paiement, MRR | Système de facturation (ex. Stripe, CRM) |
| Secteur, taille, plan | CRM commercial |
| Commentaire CSM | CRM ou outil de Customer Success dédié |

**Alternative si une source est indisponible** : à défaut de `csat` en temps réel, un
proxy possible serait le volume et le ton des tickets support ; à défaut de
`derniere_connexion_jours`, les logs d'authentification bruts pourraient être agrégés
côté infrastructure. Ce TP ne fait qu'identifier ces dépendances — la préparation
proprement dite est l'objet de TP2.

## §3 — Enjeux éthiques, sociétaux et conformité (amorce) [C2]

*Analyse préliminaire — approfondie en TP7 (biais des facteurs explicatifs) et dans le
notebook final (section dédiée du plan imposé).*

### Données personnelles (RGPD)

- `client_id` est un identifiant de compte, pas une donnée personnelle nominative.
- Aucune colonne ne contient explicitement de nom, email ou identité individuelle.
- Point de vigilance : `commentaire_csm` est un champ **texte libre** — à risque a
  priori élevé de contenir des informations nominatives. Vérification empirique
  ci-dessous.

In [4]:
n_total = len(df_complet)
n_rempli = df_complet["commentaire_csm"].notna().sum()
print(f"commentaire_csm rempli : {n_rempli}/{n_total} ({n_rempli/n_total:.1%})")
print()
print("Échantillon de valeurs distinctes :")
for v in df_complet["commentaire_csm"].dropna().unique()[:12]:
    print(" -", v)


commentaire_csm rempli : 2245/5035 (44.6%)

Échantillon de valeurs distinctes :
 - Mécontentement exprimé au support.
 - Faible adoption des sièges.
 - Client très satisfait et actif.
 - Suivi standard.
 - RAS.
 - Client insatisfait, risque de départ.
 - Compte engagé, ambassadeur potentiel.
 - Compte dans la moyenne.
 - Multiples tickets ouverts, friction élevée.
 - Déploiement interne limité.
 - Usage en forte baisse ce trimestre.
 - Compte peu actif, relance nécessaire.


Sur cet échantillon, les commentaires sont des notes qualitatives templatées
(« Client insatisfait, risque de départ. », « Faible adoption des sièges. ») — pas de
nom de personne physique identifié. Le champ reste à re-vérifier de façon systématique
en TP2 (recherche de motifs nominatifs) avant tout usage, mais le risque RGPD immédiat
apparaît limité sur ce jeu de données.

### Biais potentiels

- Le modèle utilise `secteur`, `pays`, `taille_entreprise` — des variables qui
  segmentent des **entreprises**, pas des individus. Le risque n'est pas une
  discrimination interdite (comme sur des personnes physiques), mais un **biais de
  priorisation commerciale** : si le modèle finit par systématiquement prioriser les
  gros comptes (forte CLV) au détriment de PME à risque réel, l'outil renforcerait une
  logique déjà présente côté business plutôt que de la corriger. À documenter, pas à
  "corriger" artificiellement — c'est un choix métier assumé, pas un bug.

### Conséquences d'une erreur de prédiction

| Erreur | Conséquence | Gravité |
|---|---|---|
| **Faux négatif** (rater un client qui va vraiment partir) | Perte de revenu silencieuse, aucune action de rétention déclenchée | Élevée — c'est la perte que le projet cherche à éviter |
| **Faux positif** (alerter sur un client qui allait rester) | Sollicitation commerciale/CS inutile, coût de temps | Modérée — gênant mais rarement dommageable pour le client |

Cette asymétrie (coûter cher de rater un churner vs. coûter un peu de sur-solliciter un
client fidèle) devra guider le choix du **seuil de décision** en TP6 — pas une valeur
par défaut arbitraire (0.5).

### Principe directeur retenu

Le modèle reste un **outil d'aide à la décision** : aucune action automatique (résiliation,
downgrade, blocage de compte) ne doit être déclenchée sans validation humaine par un
Customer Success Manager.

## Journal de bord — Synthèse TP1

**Confirmé** :
- Les 3 fichiers sont accessibles et cohérents avec le dictionnaire de données de
  l'énoncé (5035 × 29 pour le jeu complet).
- Deux variables suspectes identifiées à ce stade sur la seule base de leur description
  (`sante_compte_fin_periode` = piège de fuite probable ; `couleur_theme_interface`,
  `code_datacenter`, `groupe_experimentation` = leurres probables) — **à confirmer
  empiriquement en TP3**, pas encore exclues ici.
- Risque RGPD immédiat limité sur `commentaire_csm` (échantillon inspecté), à
  revérifier systématiquement en TP2.
- La CLV (`valeur_vie_client_eur`) est actée comme cible secondaire strictement séparée
  du modèle de churn.

**Reste à faire (prochains TP)** :
- TP2 — nettoyer les formats hétérogènes (dates, nombres textuels, casse) et
  documenter les valeurs manquantes.
- TP3 — confirmer statistiquement le piège de fuite et les leurres avant de les exclure.
- TP4 — EDA complète (distribution du churn, corrélations).